In [38]:
# import libraries
import numpy as np
import torch
import torch.nn as nn

# import the model
graph = np.load("model/graph.npy", allow_pickle=True)
config = np.load("model/config.npy", allow_pickle=True)
weights = np.load("model/weights.npy", allow_pickle=True)

# start the runtime
virtual_runtime = [None] * len(graph)

In [39]:
# coco classes

classes = [
    "airplane",
    "apple",
    "backpack",
    "banana",
    "baseball bat",
    "baseball glove",
    "bear",
    "bed",
    "bench",
    "bicycle",
    "bird",
    "boat",
    "book",
    "bottle",
    "bowl",
    "broccoli",
    "bus",
    "cake",
    "car",
    "carrot",
    "cat",
    "cell phone",
    "chair",
    "clock",
    "couch",
    "cow",
    "cup",
    "dining table",
    "dog",
    "donut",
    "elephant",
    "fire hydrant",
    "fork",
    "frisbee",
    "giraffe",
    "hair drier",
    "handbag",
    "horse",
    "hot dog",
    "keyboard",
    "kite",
    "knife",
    "laptop",
    "microwave",
    "motorcycle",
    "mouse",
    "orange",
    "oven",
    "parking meter",
    "person",
    "pizza",
    "potted plant",
    "refrigerator",
    "remote",
    "sandwich",
    "scissors",
    "sheep",
    "sink",
    "skateboard",
    "skis",
    "snowboard",
    "spoon",
    "sports ball",
    "stop sign",
    "suitcase",
    "surfboard",
    "teddy bear",
    "tennis racket",
    "tie",
    "toaster",
    "toilet",
    "toothbrush",
    "traffic light",
    "train",
    "truck",
    "tv",
    "umbrella",
    "vase",
    "wine glass",
    "zebra"
]

In [40]:
# Graph
# indexed by node_id, each entry is a list of input node_ids
# -1 means input from the outside (e.g., input image)

# ------------------------ #
# input node_id: ##0
# ------------------------ #
# output node_id(s):

# regression path::     <- not sure about this path but in this project we only do classification
# after dfl: 233

# classification path:: (80-logits) should convert to probabilities via softmax/sigmoid
# 1. large:  ##218
# 2. medium: ##225
# 3. small:  ##232
# ------------------------ #



print("=== Graph ===")
print(f"Type: {type(graph)}, dtype: {graph.dtype}, length: {len(graph)}")
print("First 3 entries:")
for i in range(min(3, len(graph))):
    print(f"graph[{i}] = {graph[i]}")

=== Graph ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 234
First 3 entries:
graph[0] = [-1]
graph[1] = [0]
graph[2] = [1]


In [41]:
# Config

# conv:                         [op_type, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
# bn:                           [op_type, num_features, eps, momentum, affine, track_running_stats]
# activation:                   [op_type, inplace]
# pool:                         [op_type, k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode]
# upsample:                     [op_type, scale_h, scale_w, mode]
# concat/resadd:                [op_type] (no params)

print("=== Config ===")
print(f"Type: {type(config)}, dtype: {config.dtype}, length: {len(config)}")
print("First 3 entries:")
for i in range(min(3, len(config))):
    print(f"config[{i}] = {config[i]}")

=== Config ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 234
First 3 entries:
config[0] = ['conv', 3, 16, 6, 6, 2, 2, 2, 2, False]
config[1] = ['bn', 16, 0.001, 0.03, True, True]
config[2] = ['activation', True]


In [ ]:
# Weights

# conv:                         [weight, bias(if exists)]                           <tensor>
# bn:                           [gamma, beta, running_mean, running_var]            <tensor>
# activation:                   []
# pool:                         []
# upsample:                     []
# concat:                       []  
# resadd:                       []



print("=== Weights ===")
print(f"Type: {type(weights)}, dtype: {weights.dtype}, length: {len(weights)}")


## Examples
# print("conv unbiased:", weights[0])
# print("conv biased:", weights[204])
# print("bn:", weights[1])
# print("activation:", weights[2])
# print("pool:", weights[104])
# print("upsample:", weights[112])
# print("concat:", weights[113])
# print("resadd:", weights[18])

=== Weights ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 234
conv: [tensor([[[[-9.6069e-02]],

         [[-5.3406e-02]],

         [[-2.5391e-02]],

         [[-9.0027e-03]],

         [[-1.8359e-01]],

         [[-6.0913e-02]],

         [[-7.5989e-02]],

         [[-1.3660e-01]],

         [[-6.3477e-02]],

         [[-1.8250e-01]],

         [[-1.6356e-03]],

         [[-1.7017e-01]],

         [[-3.0380e-02]],

         [[-1.0156e-01]],

         [[-6.0387e-03]],

         [[-3.1143e-02]],

         [[ 9.5062e-03]],

         [[ 1.5244e-02]],

         [[ 1.1574e-02]],

         [[-4.0359e-03]],

         [[-3.5248e-02]],

         [[-7.6790e-03]],

         [[-2.4261e-02]],

         [[-1.8787e-01]],

         [[-5.4199e-02]],

         [[-3.9520e-02]],

         [[ 1.4908e-02]],

         [[-7.7087e-02]],

         [[-3.7140e-02]],

         [[ 3.3319e-05]],

         [[-2.7954e-02]],

         [[-5.4291e-02]]],


        [[[-8.2092e-02]],

         [[-4.5715e-02]],


In [43]:
def tensor_vis(tensor, max_elements=1, indent=0):
    if not isinstance(tensor, torch.Tensor):
        raise ValueError("Input must be a torch.Tensor")
    shape = tensor.shape
    print(" " * indent + f"Tensor shape: {shape}")
    if tensor.ndim == 0:
        print(" " * indent + f"Value: {tensor.item()}")
        return
    elif tensor.ndim == 1:
        elems = tensor[:max_elements].tolist()
        if len(tensor) > max_elements:
            elems.append("...")
        print(" " * indent + str(elems))
    else:
        for i in range(min(tensor.shape[0], max_elements)):
            print(" " * indent + f"[{i}]:")
            tensor_vis(tensor[i], max_elements=max_elements, indent=indent + 2)
        if tensor.shape[0] > max_elements:
            print(" " * indent + "...")
            
print("Graph type:", type(graph))
print("Graph example (first 5 nodes):",graph)

# print("Operations type:", type(operations))
# print("Operations (first 10):", operations[:10])

# print("Config type:", type(config))
# print("Config (first 5):", config[:5])

# print("Weights type:", type(weights))
# print("Weights (first 3):", weights[:3])

Graph type: <class 'numpy.ndarray'>
Graph example (first 5 nodes): [list([-1]) list([0]) list([1]) list([2]) list([3]) list([4]) list([5])
 list([6]) list([7]) list([8]) list([9]) list([10]) list([11]) list([12])
 list([13]) list([5]) list([15]) list([16]) list([8, 14]) list([18, 17])
 list([19]) list([20]) list([21]) list([22]) list([23]) list([24])
 list([25]) list([26]) list([27]) list([28]) list([29]) list([30])
 list([31]) list([32]) list([33]) list([34]) list([35]) list([36])
 list([37]) list([38]) list([39]) list([25]) list([41]) list([42])
 list([28, 40]) list([44, 43]) list([45]) list([46]) list([47]) list([48])
 list([49]) list([50]) list([51]) list([52]) list([53]) list([54])
 list([55]) list([56]) list([57]) list([58]) list([59]) list([60])
 list([61]) list([62]) list([63]) list([64]) list([65]) list([66])
 list([67]) list([68]) list([69]) list([70]) list([71]) list([51])
 list([73]) list([74]) list([54, 72]) list([76, 75]) list([77]) list([78])
 list([79]) list([80]) list(

In [44]:
def Conv2d(config, weights):
    print("Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]")
    print("Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists]")
    in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias_flag = config
    layer = nn.Conv2d(
        in_channels=in_ch,
        out_channels=out_ch,
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        bias=bias_flag
    )
    if weights:
        layer.weight.data = weights[0]
        if bias_flag and len(weights) > 1:
            layer.bias.data = weights[1]
    return layer

def BatchNorm2d(config, weights):
    print("Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag]")
    print("Weight = [weight(num_features), bias(num_features), running_mean(num_features), running_var(num_features)]")
    num_feat, eps, momentum, affine_flag, track_stats = config
    layer = nn.BatchNorm2d(
        num_features=num_feat,
        eps=eps,
        momentum=momentum,
        affine=affine_flag,
        track_running_stats=track_stats
    )
    if weights:
        layer.weight.data = weights[0]
        layer.bias.data = weights[1]
        layer.running_mean.data = weights[2]
        layer.running_var.data = weights[3]
    return layer

def SiLU(config, weights):
    print("Config = [inplace_flag]")
    inplace_flag = config[0]
    return nn.SiLU(inplace=inplace_flag)

def MaxPool2d(config, weights):
    print("Config = [kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, ceil_mode_flag]")
    k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode = config
    return nn.MaxPool2d(
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        dilation=(d_h, d_w),
        ceil_mode=ceil_mode
    )

def Upsample(config, weights):
    print("Config = [scale_h, scale_w, mode]")
    scale_h, scale_w, mode = config
    return nn.Upsample(
        scale_factor=(scale_h, scale_w),
        mode=mode
    )

# Concat:
# Config = []
# Weight = []

def layer(operation, config, weights=None):
    if operation == "Conv2d":
        return Conv2d(config, weights)
    elif operation == "BatchNorm2d":
        return BatchNorm2d(config, weights)
    elif operation == "SiLU":
        return SiLU(config, weights)
    elif operation == "MaxPool2d":
        return MaxPool2d(config, weights)
    elif operation == "Upsample":
        return Upsample(config, weights)
    elif operation == "Concat":
        raise ValueError(f"Unsupported layer type: {operation}")
    else:
        raise ValueError(f"Unsupported layer type: {operation}")


In [45]:
from PIL import Image
import torch
from torchvision import transforms

def preprocess(image_path, img_size=(640, 640), device='cpu'):
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    elif isinstance(image_path, Image.Image):
        img = image_path.convert("RGB")
    else:
        raise TypeError("image_path must be a file path or PIL.Image")

    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
    ])

    tensor = transform(img).unsqueeze(0).to(device)
    return tensor

In [46]:
from PIL import Image
frame = Image.open("/mnt/fileserver/prj/yolo-inference-from-scratch/imgs/000000000785.jpg")
frame = preprocess(frame)
tensors.append(frame)

NameError: name 'tensors' is not defined

In [ ]:
def step(l):
    print(f"\n>>> Layer {l} Info <<<")    
    print(f"\n-----in-----")
    current_layer = layer(operations[l], config[l], weights[l])
    input_tensor = tensors[graph[l][0]+1]
    print("layert:", operations[l])
    print(f"Input tensor shape: {input_tensor.shape}")
    print(f"Layer config: {config[l]}")
    print(f"Graph mapping: {graph[l]}")
    if weights[l]:
        for i, ts in enumerate(weights[l]):
            print(f"\nWeight tensor {i}:")
            tensor_vis(ts)
    
    output_tensor = current_layer(input_tensor)
    tensors.append(output_tensor)


    print(f"\n-----out-----")
    print("\nOutput tensor:")
    tensor_vis(output_tensor)

In [ ]:
for l in range(400):
    try:
        step(l)
    except RuntimeError as e:
        print(f"\n>>> Layer {l} skipped due to error:")
        print(e)
        if tensors:
            for i, ts in enumerate(tensors):
                print(f"\nTensors {i}:")
                tensor_vis(ts)


In [ ]:
print(graph)